In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import os
import pandas as pd
import gc
from pathlib import Path
import netCDF4 as nc
from datetime import datetime
import re

In [2]:
gemlam_dir = "/results/forcing/atmospheric/GEM2.5/gemlam"
operational_dir = "/results/forcing/atmospheric/GEM2.5/operational"
time_fixed_dir = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/TimeFixed"

gemlam_start = datetime(2007, 1, 3)
gemlam_end = datetime(2014, 9, 30)
ops_start = datetime(2014, 10, 1)
ops_end = datetime(2021, 12, 31)

years = range(2007, 2022)

fixed_2008_filenames = {
    "gemlam_y2008m07d16.nc",
    "gemlam_y2008m07d17.nc",
    "gemlam_y2008m07d18.nc",
    "gemlam_y2008m07d19.nc",
    "gemlam_y2008m07d20.nc",
    "gemlam_y2008m07d21.nc",
    "gemlam_y2008m07d22.nc",
    "gemlam_y2008m07d23.nc",
    "gemlam_y2008m08d10.nc",
    "gemlam_y2008m08d11.nc",
    "gemlam_y2008m08d12.nc",
}

def get_file_date(filepath):
    filename = os.path.basename(filepath)
    match = re.search(r"_y(\d{4})m(\d{2})d(\d{2})", filename)
    if match is None:
        return None
    year, month, day = map(int, match.groups())
    return datetime(year, month, day)

all_gemlam_files = glob.glob(os.path.join(gemlam_dir, "*.nc"))
all_ops_files = glob.glob(os.path.join(operational_dir, "ops_y????m??d??.nc"))

selected_gemlam_files = {}

for filepath in all_gemlam_files:
    file_date = get_file_date(filepath)
    if file_date is not None and gemlam_start <= file_date <= gemlam_end:
        selected_gemlam_files[file_date] = filepath

for filename in fixed_2008_filenames:
    fixed_filepath = os.path.join(time_fixed_dir, filename)
    if not os.path.exists(fixed_filepath):
        raise FileNotFoundError(f"Corrected file not found: {fixed_filepath}")
    fixed_date = get_file_date(fixed_filepath)
    if fixed_date is None:
        raise ValueError(f"Could not extract date from: {fixed_filepath}")
    selected_gemlam_files[fixed_date] = fixed_filepath

selected_ops_files = {}

for filepath in all_ops_files:
    file_date = get_file_date(filepath)
    if file_date is not None and ops_start <= file_date <= ops_end:
        selected_ops_files[file_date] = filepath

all_selected_files = {**selected_gemlam_files, **selected_ops_files}

hrdps_files_by_year = {}

for year in years:
    files = [filepath for file_date, filepath in sorted(all_selected_files.items()) if file_date.year == year]
    if len(files) == 0:
        print(f"{year}: no files found")
        continue
    hrdps_files_by_year[year] = files
    print(year, ":", files[0], "to", files[-1], f"({len(files)} files)")

2007 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2007m01d03.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2007m12d31.nc (363 files)
2008 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2008m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2008m12d31.nc (366 files)
2009 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2009m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2009m12d31.nc (365 files)
2010 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2010m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2010m12d31.nc (365 files)
2011 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2011m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2011m12d31.nc (365 files)
2012 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2012m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2012m12d31.nc (366 files)
2013 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2013m01d01.nc to /results/forc

In [3]:
# Loading datasets

weights_pre_file = "/home/sallen/MEOPAR/grid/weights-gem2.5-gemlam_201702_pre22sep11.nc"
weights_post_file = "/home/sallen/MEOPAR/grid/weights-gem2.5-gemlam_201702_22sep11onward.nc"
mesh_mask_file = "/ocean/dtaneja/MOAD/analysis-dishika/grid/mesh_mask202108.nc"

ds_weights_pre = xr.open_dataset(weights_pre_file).load()
ds_weights_post = xr.open_dataset(weights_post_file).load()

with xr.open_dataset(mesh_mask_file) as ds_mesh:
    nemo_lat = ds_mesh["nav_lat"].load()
    nemo_lon = ds_mesh["nav_lon"].load()

In [4]:
# Converts hourly HRDPS temperature from the 266 × 256 HRDPS grid to the 898 × 398 NEMO grid

transition_time = pd.Timestamp("2011-09-22 00:00:00")

def interpolate_therm_rad_with_weights(therm_rad, ds_weights):
    therm_rad = therm_rad.transpose("time_counter", "y", "x")

    n_time = therm_rad.sizes["time_counter"]
    source_shape = (therm_rad.sizes["y"],therm_rad.sizes["x"])
    n_source_cells = source_shape[0] * source_shape[1]

    source_values = (therm_rad.values.reshape(n_time, n_source_cells).astype(np.float32))
    target_shape = ds_weights["src01"].shape

    interpolated = np.zeros((n_time, target_shape[0], target_shape[1]),dtype=np.float32)

    for n in range(1, 5):
        # The source indexes in the weights file are 1-based
        source_index = (ds_weights[f"src{n:02d}"].values.astype(np.int64)- 1)
        weight = (ds_weights[f"wgt{n:02d}"].values.astype(np.float32))

        interpolated += (source_values[:, source_index]* weight[None, :, :])

    therm_rad_nemo = xr.DataArray(
        interpolated,
        dims=("time_counter", "y", "x"),
        coords={
            "time_counter": therm_rad["time_counter"],
            "nav_lat": (("y", "x"), nemo_lat.values),
            "nav_lon": (("y", "x"), nemo_lon.values),
        },
        name="therm_rad"
    )

    therm_rad_nemo.attrs = therm_rad.attrs.copy()
    therm_rad_nemo.attrs["grid"] = "SalishSeaCast NEMO grid"
    therm_rad_nemo.attrs["interpolation"] = (
        "Four-source weighted HRDPS-to-NEMO interpolation"
    )
    return therm_rad_nemo

In [5]:
# Open file and interpolates

def extract_and_interpolate_hourly_therm_rad(file):
    with xr.open_dataset(file) as ds:
        therm_rad = ds["therm_rad"].sortby("time_counter")
        time_index = therm_rad.get_index("time_counter")

        if time_index.has_duplicates:
            unique_mask = ~time_index.duplicated()
            therm_rad = therm_rad.isel(time_counter=unique_mask)

        therm_rad = therm_rad.load()

    times = pd.to_datetime(therm_rad["time_counter"].values)

    before_transition = times < transition_time
    after_transition = times >= transition_time

    pieces = []

    if before_transition.any():
        therm_rad_pre = therm_rad.isel(time_counter=np.where(before_transition)[0])
        interpolated_pre = interpolate_therm_rad_with_weights(therm_rad_pre,ds_weights_pre)
        pieces.append(interpolated_pre)

    if after_transition.any():
        therm_rad_post = therm_rad.isel(time_counter=np.where(after_transition)[0])
        interpolated_post = interpolate_therm_rad_with_weights(therm_rad_post,ds_weights_post)
        pieces.append(interpolated_post)

    therm_rad_nemo = xr.concat(pieces,dim="time_counter").sortby("time_counter")
    return therm_rad_nemo

In [6]:
sample_pre_file = hrdps_files_by_year[2008][0]
therm_rad_nemo_pre = extract_and_interpolate_hourly_therm_rad(sample_pre_file)

print("Shape:", therm_rad_nemo_pre.shape)
print("Time range:",therm_rad_nemo_pre.time_counter.values[0],"to",therm_rad_nemo_pre.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2008-01-01T00:00:00.000000000 to 2008-01-01T23:00:00.000000000


In [7]:
sample_post_file = hrdps_files_by_year[2012][0]
therm_rad_nemo_post = extract_and_interpolate_hourly_therm_rad(sample_post_file)

print("Shape:", therm_rad_nemo_post.shape)
print("Time range:",therm_rad_nemo_post.time_counter.values[0],"to",therm_rad_nemo_post.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2012-01-01T00:00:00.000000000 to 2012-01-01T23:00:00.000000000


In [8]:
sample_post_file = hrdps_files_by_year[2015][0]
therm_rad_nemo_post = extract_and_interpolate_hourly_therm_rad(sample_post_file)

print("Shape:", therm_rad_nemo_post.shape)
print("Time range:",therm_rad_nemo_post.time_counter.values[0],"to",therm_rad_nemo_post.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2015-01-01T00:00:00.000000000 to 2015-01-01T23:00:00.000000000


In [9]:
sample_post_file = hrdps_files_by_year[2021][-1]
therm_rad_nemo_post = extract_and_interpolate_hourly_therm_rad(sample_post_file)

print("Shape:", therm_rad_nemo_post.shape)
print("Time range:",therm_rad_nemo_post.time_counter.values[0],"to",therm_rad_nemo_post.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2021-12-31T00:00:00.000000000 to 2021-12-31T23:00:00.000000000


In [10]:
with xr.open_dataset(mesh_mask_file) as ds_mesh:
    water_mask = (ds_mesh["tmask"].isel(t=0, z=0).load().values.astype(bool))
    nemo_lat_2d = ds_mesh["nav_lat"].load().values
    nemo_lon_2d = ds_mesh["nav_lon"].load().values
nemo_j, nemo_i = np.where(water_mask)
water_flat_indices = np.flatnonzero(water_mask.reshape(-1))
nemo_water_lat = nemo_lat_2d[water_mask]
nemo_water_lon = nemo_lon_2d[water_mask]
n_water = len(water_flat_indices)
print("NEMO grid shape:", water_mask.shape)
print("Number of surface water cells:", n_water)

NEMO grid shape: (898, 398)
Number of surface water cells: 81383


In [11]:
def process_daily_file_to_3h_water(file):
    therm_rad_nemo_hourly = extract_and_interpolate_hourly_therm_rad(file)
    n_time = therm_rad_nemo_hourly.sizes["time_counter"]

    therm_rad_water_values = (therm_rad_nemo_hourly.values.reshape(n_time, -1)[:, water_flat_indices].astype(np.float32))
    therm_rad_water_hourly = xr.DataArray(therm_rad_water_values,dims=("time_counter", "water_cell"),coords={
            "time_counter": therm_rad_nemo_hourly["time_counter"].values,
            "water_cell": np.arange(n_water, dtype=np.int32),},name="therm_rad",attrs=therm_rad_nemo_hourly.attrs,)

    therm_rad_water_3h = therm_rad_water_hourly.resample(time_counter="3h",label="left",closed="left",origin="start_day",).mean()
    return therm_rad_water_3h.load()

In [12]:
test_file = hrdps_files_by_year[2008][0]

test_3h = process_daily_file_to_3h_water(test_file)

print(test_3h)
print("Shape:", test_3h.shape)
print("Times:", test_3h.time_counter.values)

<xarray.DataArray 'therm_rad' (time_counter: 8, water_cell: 81383)> Size: 3MB
array([[254.25856, 254.50563, 254.80165, ..., 212.44531, 211.2187 ,
        209.82051],
       [254.0562 , 254.50995, 255.03642, ..., 263.59375, 261.60287,
        259.37183],
       [282.40344, 283.60526, 284.91504, ..., 282.67923, 281.46408,
        280.12332],
       ...,
       [270.80984, 271.97336, 273.18423, ..., 291.6092 , 291.0622 ,
        290.4569 ],
       [313.41724, 313.37048, 313.29858, ..., 296.4727 , 296.3238 ,
        296.16672],
       [327.19403, 327.20743, 327.24356, ..., 301.322  , 301.7306 ,
        302.15787]], shape=(8, 81383), dtype=float32)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 64B 2008-01-01 ... 2008-01-01...
  * water_cell    (water_cell) int32 326kB 0 1 2 3 4 ... 81379 81380 81381 81382
Attributes:
    level:          surface
    long_name:      Downward Long-Wave Radiation Flux
    standard_name:  net_downward_longwave_flux_in_air
    units:          W/m^2

In [13]:
test_file = hrdps_files_by_year[2015][0]

test_3h = process_daily_file_to_3h_water(test_file)

print(test_3h)
print("Shape:", test_3h.shape)
print("Times:", test_3h.time_counter.values)

<xarray.DataArray 'therm_rad' (time_counter: 8, water_cell: 81383)> Size: 3MB
array([[233.95558, 233.99687, 234.05334, ..., 229.399  , 229.72241,
        230.02856],
       [235.44325, 235.52718, 235.57683, ..., 234.88823, 234.2298 ,
        233.52002],
       [235.00371, 235.12886, 235.23503, ..., 231.8751 , 231.08516,
        230.23796],
       ...,
       [231.62767, 231.84029, 232.02364, ..., 234.39697, 233.94464,
        233.46947],
       [236.49904, 236.5417 , 236.53487, ..., 295.1852 , 295.37665,
        295.53406],
       [240.46375, 240.43633, 240.49983, ..., 266.1715 , 266.40652,
        266.5443 ]], shape=(8, 81383), dtype=float32)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 64B 2015-01-01 ... 2015-01-01...
  * water_cell    (water_cell) int32 326kB 0 1 2 3 4 ... 81379 81380 81381 81382
Attributes:
    short_name:     DLWRF_surface
    long_name:      Downward Long-Wave Rad. Flux
    level:          surface
    units:          W/m^2
    grid:           Sali

In [14]:
# # Run only once for the remaining years
# output_dir = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/data/hrdps_nemo_3h"
# os.makedirs(output_dir, exist_ok=True)
# remaining_years = [2007] + list(range(2013, 2022))
# hrdps_nemo_processed_files = []

# for year in remaining_years:
#     files = hrdps_files_by_year[year]
#     print(f"\nProcessing {year}: {len(files)} daily files")
#     daily_3h_results = []

#     for file_number, file in enumerate(files, start=1):
#         daily_3h = process_daily_file_to_3h_water(file)
#         daily_3h_results.append(daily_3h)

#         if file_number == 1 or file_number % 25 == 0 or file_number == len(files):
#             print(f"{year}: processed {file_number}/{len(files)} files")

#     therm_rad_year = xr.concat(daily_3h_results, dim="time_counter").sortby("time_counter")
#     time_index = therm_rad_year.get_index("time_counter")

#     if time_index.has_duplicates:
#         therm_rad_year = therm_rad_year.isel(time_counter=~time_index.duplicated())

#     ds_year = therm_rad_year.to_dataset(name="therm_rad")
#     ds_year = ds_year.assign_coords(nemo_j=("water_cell", nemo_j.astype(np.int32)), nemo_i=("water_cell", nemo_i.astype(np.int32)), nav_lat=("water_cell", nemo_water_lat.astype(np.float32)), nav_lon=("water_cell", nemo_water_lon.astype(np.float32)))
#     ds_year.attrs["description"] = "Raw hourly HRDPS therm_rad interpolated onto NEMO surface water cells, then resampled to three-hourly means."

#     output_file = f"{output_dir}/HRDPS_NEMO_{year}_therm_rad_3h.nc"
#     encoding = {"therm_rad": {"dtype": "float32", "zlib": True, "complevel": 4}}
#     ds_year.to_netcdf(output_file, engine="netcdf4", encoding=encoding)
#     hrdps_nemo_processed_files.append(output_file)

#     print("Saved:", output_file)
#     print("Shape:", ds_year["therm_rad"].shape)
#     print("Time range:", ds_year.time_counter.values[0], "to", ds_year.time_counter.values[-1])

#     del daily_3h_results
#     del therm_rad_year
#     del ds_year
#     gc.collect()

In [15]:
hrdps_nemo_files = sorted(glob.glob("/ocean/dtaneja/MOAD/analysis-dishika/notebooks/data/hrdps_nemo_3h/HRDPS_NEMO_*_therm_rad_3h.nc"))
ds_hrdps_nemo = xr.open_mfdataset(hrdps_nemo_files,combine="by_coords")

ds_hrdps_nemo = ds_hrdps_nemo.sortby("time_counter")

print(ds_hrdps_nemo)
print("First time:", ds_hrdps_nemo.time_counter.values[0])
print("Last time: ", ds_hrdps_nemo.time_counter.values[-1])
print("Shape:", ds_hrdps_nemo["therm_rad"].shape)

<xarray.Dataset> Size: 14GB
Dimensions:       (time_counter: 43808, water_cell: 81383)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 350kB 2007-01-03 ... 2021-12-...
  * water_cell    (water_cell) int32 326kB 0 1 2 3 4 ... 81379 81380 81381 81382
    nemo_j        (water_cell) int32 326kB dask.array<chunksize=(81383,), meta=np.ndarray>
    nemo_i        (water_cell) int32 326kB dask.array<chunksize=(81383,), meta=np.ndarray>
    nav_lat       (water_cell) float32 326kB dask.array<chunksize=(81383,), meta=np.ndarray>
    nav_lon       (water_cell) float32 326kB dask.array<chunksize=(81383,), meta=np.ndarray>
Data variables:
    therm_rad     (time_counter, water_cell) float32 14GB dask.array<chunksize=(363, 10173), meta=np.ndarray>
Attributes:
    description:  Raw hourly HRDPS therm_rad interpolated onto NEMO surface w...
First time: 2007-01-03T00:00:00.000000000
Last time:  2021-12-31T21:00:00.000000000
Shape: (43808, 81383)


In [16]:
# Validation: 2007–2009
ds_hrdps_val = ds_hrdps_nemo.sel(time_counter=slice("2007-01-01", "2009-12-31 23:59:59"))
# Training: 2010–2018
ds_hrdps_train = ds_hrdps_nemo.sel(time_counter=slice("2010-01-01", "2018-12-31 23:59:59"))
# Testing: 2019–2021
ds_hrdps_test = ds_hrdps_nemo.sel(time_counter=slice("2019-01-01", "2021-12-31 23:59:59"))

In [17]:
canrcm_years = [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
canrcm_rlds_files = []

for year in canrcm_years:
    matches = sorted(glob.glob(f"/results/forcing/CanRCM5/*_{year}01_{year}12_3h_rlds.nc"))
    canrcm_rlds_files.append(matches[0])

print("CanRCM files:")
for file in canrcm_rlds_files:
    print(file)

CanRCM files:
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200801_200812_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200901_200912_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201001_201012_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201101_201112_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201201_201212_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201301_201312_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201401_201412_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201501_201512_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201601_201612_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201701_201712_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201801_201812_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_201901_201912_3h_rlds.nc
/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_202001_202012_3h_

In [18]:
from datetime import timedelta
ds_canrcm = xr.open_mfdataset(canrcm_rlds_files,combine="by_coords")
ds_canrcm = ds_canrcm.sortby("time")
shifted_time = np.array([time_value - timedelta(hours=3)for time_value in ds_canrcm["time"].values])
ds_canrcm = ds_canrcm.assign_coords(time=("time", shifted_time))

print(ds_canrcm)
print("First timestamp:", ds_canrcm.time.values[0])
print("Last timestamp: ", ds_canrcm.time.values[-1])

/tmp/ipykernel_2257576/2851013810.py:2: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_canrcm = xr.open_mfdataset(canrcm_rlds_files,combine="by_coords")


<xarray.Dataset> Size: 20GB
Dimensions:       (time: 43800, bnds: 2, rlat: 320, rlon: 360)
Coordinates:
  * time          (time) object 350kB 2007-01-01 00:00:00 ... 2021-12-31 21:0...
  * rlat          (rlat) float64 3kB -35.11 -34.89 -34.67 ... 34.63 34.85 35.07
  * rlon          (rlon) float64 3kB -39.27 -39.05 -38.83 ... 39.27 39.49 39.71
    lon           (rlat, rlon) float64 922kB dask.array<chunksize=(320, 360), meta=np.ndarray>
    lat           (rlat, rlon) float64 922kB dask.array<chunksize=(320, 360), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    time_bnds     (time, bnds) object 701kB dask.array<chunksize=(2920, 2), meta=np.ndarray>
    rotated_pole  (time) |S1 44kB b'' b'' b'' b'' b'' ... b'' b'' b'' b'' b''
    rlds          (time, rlat, rlon) float32 20GB dask.array<chunksize=(2920, 320, 360), meta=np.ndarray>
Attributes: (12/17)
    title:                          CanRCM4 model output prepared for CORDEX ...
    institution:                  